# Raschig Super-Ring #0.3 (PLAN 4b, second packing)
A random (dumped) packing, so a different family from both Flexipac and Super-Pak. Data: Wang (2015) Table C.1, SRP column (0.427 m, air/water).
All the model code lives in `../packlib.py` (D47). This notebook only loads the data, sets the properties and calls the library.
Decisions used: D49 and D53 (fluid properties), D54 (Level 0 constants, curves), D55 (dry rows).

In [ ]:
# 1.1 load the library and the data

import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import packlib

df = pd.read_csv('../srp_data/wang2015_tableC1.csv')
data = df[df['packing'] == 'RSR#0.3'].copy()
print(data.shape)
print(data['run_header'].unique())

In [ ]:
# 1.2 water properties per row from each row's liquid temperature (D49)
# nist webbook table, 0 to 40 C (liquid here runs to 32 C), straight-line interpolation between rows

water = pd.read_csv('../srp_data/properties/nist_water_0_40C.csv')
data['rho_L'] = np.interp(data['Tliq_in_C'], water['T_C'], water['rho_kg_m3'])
data['mu_L'] = np.interp(data['Tliq_in_C'], water['T_C'], water['mu_Pa_s'])

print(data[['Tliq_in_C', 'rho_L', 'mu_L']].describe().round(5))

In [ ]:
# 1.3 air properties per row at the mean of air in and air out (D53)
# density: ideal gas, dry air, 101.325 kPa assumed (D53)
# viscosity: sutherland's law with white's constants (viscous fluid flow, table 1-2)

P = 101325        # Pa, assumed
M_air = 0.0289644 # kg/mol, dry air (iso 2533 standard atmosphere)
R = 8.314462      # J/mol/K

T_gas = (data['Tair_in_C'] + data['Tair_out_C']) / 2 + 273.15
data['rho_G'] = P * M_air / (R * T_gas)
data['mu_G'] = 1.716e-5 * (T_gas / 273.15) ** 1.5 * (273.15 + 110.4) / (T_gas + 110.4)

print(data[['rho_G', 'mu_G']].describe().round(7))

In [ ]:
# 1.4 standard columns the library expects

data['system'] = 'air/water'
data['F_Pa05'] = data['FG_Pa05']
data['uG_ms'] = data['F_Pa05'] / np.sqrt(data['rho_G'])
data['dpdz_Pa_m'] = data['dPdz_Pa_m']
data['holdup'] = data['holdup_frac']

# dry rows. Cp is fitted without the lowest-F dry row, the rule for every packing (D55)
dry = data[data['L_m3_m2h'] == 0].copy()
dry_fit = dry.drop(dry['F_Pa05'].idxmin())
wet = data[data['L_m3_m2h'] > 0].copy()

# any zero hold-up on a wet row can't be real, it becomes missing (same rule as D51, none expected here)
wet.loc[wet['holdup'] <= 0, 'holdup'] = np.nan

print(dry.shape, dry_fit.shape, wet.shape, wet['holdup'].isna().sum())

In [ ]:
# 1.5 curves by nominal liquid load (same rule as D50, D54). billet still uses each row's own load

nominal = np.array([12.2, 24.5, 36.6, 48.9, 61.2, 73.4])
wet['uL_m3_m2h'] = wet['L_m3_m2h']
wet['curve_load'] = [nominal[np.abs(nominal - L).argmin()] for L in wet['L_m3_m2h']]

curve = wet[['system', 'curve_load', 'uL_m3_m2h', 'F_Pa05', 'uG_ms', 'dpdz_Pa_m', 'holdup',
             'rho_L', 'mu_L', 'rho_G', 'mu_G']].copy()
curve = curve.sort_values(['curve_load', 'F_Pa05']).reset_index(drop=True)

# the library groups rows into curves by curve_id when it's there
curve['curve_id'] = curve['system'] + ' ' + curve['curve_load'].astype(str)
print(curve.groupby('curve_id').size())
print(curve.groupby('curve_load')['uL_m3_m2h'].agg(['min', 'max']))

In [ ]:
# 2.1 settings. geometry: wang table 5.2, matches billet and schultes 1999 table 2a (a 315, eps 0.96)
# level 0 constants: billet and schultes 1999 table 2a, raschig super-ring metal 0.3 (D54)
# reference water for billet's flood hold-up: nist at 20 C (D53)
# rho_G, mu_G and props below are only fallbacks, every row carries its own (D49)

C_bs = {'Cp': 0.760, 'Clp': 3.560, 'Cfp': 2.340, 'Ch': 0.750}

s = packlib.make_settings(ap=315, eps=0.96, dc=0.427,
                          rho_G=curve['rho_G'].mean(), mu_G=curve['mu_G'].mean(),
                          props={'air/water': (curve['rho_L'].mean(), curve['mu_L'].mean())},
                          C0=C_bs, dry=dry_fit, rho_W=998.21, mu_W=1.0014e-3)
print(s['ap'], s['eps'], s['dc'], s['C0'])

In [ ]:
# 2.2 level 0 on every row, and the dry rows with the published Cp

pred0 = packlib.billet_rows(curve, C_bs, s)
dp_err = 100 * (pred0['dp'] - curve['dpdz_Pa_m']) / curve['dpdz_Pa_m']
hold_err = 100 * (pred0['hold'] - curve['holdup']) / curve['holdup']
print('answered', pred0['dp'].notna().sum(), 'of', len(curve))
print('dp abs / bias %', round(dp_err.abs().mean(), 1), round(dp_err.mean(), 1))
print('hold abs / bias %', round(hold_err.abs().mean(), 1), round(hold_err.mean(), 1))

s_dry = dict(s)
s_dry['rho_G'] = dry['rho_G'].values
s_dry['mu_G'] = dry['mu_G'].values
dry_err = 100 * (packlib.dry_dp(dry['uG_ms'].values, C_bs, s_dry) - dry['dpdz_Pa_m'].values) / dry['dpdz_Pa_m'].values
print('dry dp abs / bias %', round(np.abs(dry_err).mean(), 1), round(dry_err.mean(), 1))

In [ ]:
# 2.3 level 1 fitted on every row, to see the constants

lib = packlib.fit_constants(curve, ['hold'], s)
print({k: round(v, 4) for k, v in lib['air/water'].items()})

In [ ]:
# 3.1 the tests. one liquid, so no liquid-transfer test. curves come from curve_id (D54)

splits = packlib.make_splits(curve)
for name, pairs in splits.items():
    print(name, '|', len(pairs), 'folds |', sum(len(test) for train, test in pairs), 'test rows')

In [ ]:
# 3.2 level 0, level 1 and the two fixed level 2 forms on every test

tests = packlib.run_tests(splits, s, ['L0', 'L1', 'L2 linear', 'L2 network'])

In [ ]:
# 3.3 the auto-tuned level 2 (D46) on leave one curve out only, it refits many times per fold

tuned = packlib.run_tests({'leave one curve out': splits['leave one curve out']}, s, ['L2 tuned'])
tests['leave one curve out']['L2 tuned'] = tuned['leave one curve out']['L2 tuned']

In [ ]:
# 3.4 one table, errors on the rows every model answers

results = packlib.results_table(tests, curve)
results.round(1)

In [ ]:
# 3.5 spread (D34): level 1 against level 0, and level 2 against level 1, by curve. negative = better

print(packlib.spread_table(tests, curve, 'L0', ['L1']).round(1))
packlib.spread_table(tests, curve, 'L1', ['L2 linear', 'L2 network', 'L2 tuned']).round(1)